# Aurora 0.25° small — DIMER E2E weather-forecasting fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/aurora-earth-system-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/aurora-earth-system-pipeline/blob/main/tutorials/aurora_earth_system_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-microsoft%2Faurora-ffcc4d?style=flat)](https://huggingface.co/microsoft/aurora) [![Upstream](https://img.shields.io/badge/Upstream-microsoft%2Faurora-181717?style=flat&logo=github&logoColor=white)](https://github.com/microsoft/aurora) [![Paper](https://img.shields.io/badge/Nature-10.1038%2Fs41586--025--09005--y-b31b1b.svg)](https://doi.org/10.1038/s41586-025-09005-y)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** 6-hourly global weather forecasting from two gridded analyses, lead-time evaluation against persistence, and bounded LoRA fine-tuning of the foundation model

**This notebook is standalone.** It carries the repository's package (3 modules under `src/aurora_earth_system_pipeline/`, at revision `uncommitted`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `a96afd7ee6d65e3bd2d476f3be798a25a56f2296` (~464 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies (torch, microsoft-aurora, timm, einops, xarray, netCDF4, numcodecs, numpy, safetensors, huggingface-hub), stages and digest-verifies the pinned Aurora small checkpoint (451 MB) and static fields (12 MB) from the Hub, statically audits both pickles against allow-lists, converts them once into safetensors with pinned digests, rebuilds the model from the installed package and loads it strictly with zero-initialised LoRA parameters, fetches four two-day windows of real ERA5 reanalysis at 1.5° from WeatherBench 2 (about 196 MB of pinned, digest-verified Zarr chunks — no credential), validates them and assigns them to training, validation and test roles, rolls the frozen model out to 24 h and scores it per variable and lead time against the persistence baseline, runs a bounded LoRA fine-tuning on one-step forecasts, scores the held-out window again at every lead, forecasts from a new origin, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify forecast parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about six minutes of model time after the downloads.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own gridded analyses as a NetCDF file (coordinates `time`, `level`, `latitude`, `longitude`; surface variables `2t`, `10u`, `10v`, `msl` as (time, lat, lon); atmospheric variables `t`, `u`, `v`, `q`, `z` on the 13 standard levels as (time, level, lat, lon); static fields `static_lsm`, `static_z`, `static_slt` as (lat, lon); at least three 6-hourly steps on a global equiangular grid). Your window becomes the training, validation and test window of the same contract — validation, baselines, adaptation, held-out evaluation, inference, artifact export and reload parity — and the notebook says when a single window makes the split degenerate. The expected schema, the grid rules and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

Aurora is a foundation model for the Earth system (Bodnar et al., Nature 2025): a 3D Swin transformer between a Perceiver encoder and decoder that takes two consecutive global analyses — four surface variables, five atmospheric variables on 13 pressure levels, three static fields — and returns the state six hours later; rolling that step forward gives a forecast. The checkpoint here is the **0.25° small pretrained** variant (113 M parameters), which the upstream authors publish for debugging and testing; the production checkpoints are 1.3 B parameters and are not packaged here.

Two things about this row are handled in the open. **Both upstream assets are pickles.** Section 3 downloads and digest-verifies them, statically lists every global each pickle would import (a state dict of tensors; three numpy arrays), refuses anything outside those allow-lists, unpickles each exactly once through a restricted loader, and writes safetensors whose digests are pinned in the carried module. The model you run is rebuilt from the installed `microsoft-aurora` package and loads those files strictly. **The data is real ERA5 at 1.5°, not the model's native 0.25°.** Four two-day windows come from WeatherBench 2's public bucket as pinned, digest-verified Zarr chunks — a native window would be sixty times larger — so the frozen model runs six times coarser than it was trained, an out-of-distribution use. That is the honest setting for the adaptation contract: Section 6 shows the frozen model losing to persistence at this resolution, and Section 8 shows what a 540 k-parameter LoRA fine-tuning on twelve one-step forecasts recovers.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify two pickled upstream assets, read their static audits and see them converted into safetensors; fetch and validate real gridded reanalysis from pinned objects; roll a foundation weather model out to 24 h and read latitude-weighted RMSE per variable and lead time against persistence; run a bounded LoRA fine-tuning with explicit hyperparameters; evaluate the adapted model on an independent window at every lead; forecast from a new origin; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** the 0.25° native-resolution path and the 1.3 B-parameter production checkpoints, the wave, air-pollution and 0.1° variants, ensemble forecasting, tropical-cyclone tracking, multi-step (roll-out) fine-tuning, climatology and operational-forecast baselines, the published WeatherBench scores, and any claim that a 1.5° two-day window stands in for an operational evaluation. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is enough — one 6-hour step at 1.5° takes about 0.7 s and the default fine-tuning about four minutes — and CUDA is used automatically when present. About 2 GB of RAM is needed for the model and the four windows.
- **Knowledge:** what a gridded atmospheric analysis is (pressure levels, surface fields, an equiangular grid), what a lead time and a persistence forecast are, and how RMSE is read.
- **Executable serialization handled explicitly:** the pinned checkpoint and static file are pickles. Each is digest-verified, statically audited against an allow-list (audit digests pinned) and unpickled **once** through a restricted loader to produce the safetensors the model is actually loaded from. No Hub-hosted Python module is imported; `microsoft-aurora` is installed from PyPI at a pinned version.
- **Data contract:** a window is `{{lat, lon, levels, times, surf, atmos, static}}` on a global equiangular grid — latitudes spanning 90 to −90 (17..721 rows, a multiple of 4 or one more), longitudes covering 0 to 360 (32..1440 columns, a multiple of 4), exactly the 13 standard pressure levels, 3..64 analyses spaced exactly 6 h apart, SI units (K, m/s, Pa, kg/kg, m²/s²) within plausibility ranges. The model predicts the rows the patch size covers (121 → 120 at 1.5°). BYOD accepts NetCDF in the shape Section 4 writes.
- **Validation is structural, not meteorological:** nothing checks that the fields are dynamically consistent, that the analysis is real, or that the resolution is one the model was trained on — a smooth random field within range is forecast without complaint.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — a proprietary analysis or an embargoed forecast dataset is exactly that. The default path uploads nothing.
- **External access (data):** besides the Hub, the default path fetches 57 pinned objects (about 196 MB) from the public WeatherBench 2 bucket `storage.googleapis.com/weatherbench2` over HTTPS, digest-verified before decoding; ERA5 is © ECMWF/Copernicus under the licence to use Copernicus products.
- **External access:** the Hugging Face Hub only, to fetch the pinned `microsoft/aurora` snapshot (~464 MB in total) at revision `a96afd7ee6d6…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `timm`, `xarray`, `numcodecs` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'microsoft-aurora==2.0.1',
    'timm==1.0.29',
    'einops==0.8.2',
    'xarray==2026.7.0',
    'netCDF4==1.7.4',
    'numcodecs==0.17.0',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'aurora-earth-system-pipeline',
    'repository_revision': 'uncommitted',
    'embedded_module': 'src/aurora_earth_system_pipeline/pipeline.py',
    'embedded_modules': ['src/aurora_earth_system_pipeline/pipeline.py', 'src/aurora_earth_system_pipeline/metrics.py', 'src/aurora_earth_system_pipeline/samples.py'],
    'module_sha256': '3b46d55bafe24a3a6004ad5e64f3ed77c6bf0d19ae5552cd12abc42b0bfd3840',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, timm, xarray, numcodecs
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'xarray': xarray.__version__, 'numcodecs': numcodecs.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/aurora_earth_system_pipeline/` @ `uncommitted`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/aurora_earth_system_pipeline/pipeline.py`

In [ ]:
"""Aurora 0.25° small pretrained (`microsoft/aurora`) DIMER pipeline: verified snapshot, 6-hourly global
weather forecasting from two analysis steps, lead-time evaluation against persistence, and bounded LoRA
fine-tuning of the foundation model to a user's gridded reference data with a portable adapter.

Aurora is a foundation model for the Earth system (Bodnar et al., Nature 2025): a 3D Swin transformer
backbone between a Perceiver encoder and decoder that maps two consecutive atmospheric states — four
surface variables, five atmospheric variables on pressure levels, three static fields — on an
equiangular latitude/longitude grid to the state six hours later. The "0.25° small pretrained"
checkpoint packaged here is the 113 M-parameter variant the upstream authors publish for debugging and
testing; the 1.3 B-parameter production checkpoints are the same architecture at larger width.

Two upstream assets are pickles. Under the fleet asset specification (§11) that is executable
serialization, so this package converts both once and serves neither:

* `aurora-0.25-small-pretrained.ckpt` is a torch archive whose pickle references only
  `collections.OrderedDict` and torch's tensor-rebuild helpers (verified statically by
  `audit_pickle`); it is loaded with `torch.load(weights_only=True)` — torch's restricted unpickler —
  adapted to the current parameter naming with the upstream compatibility shim, loaded into
  `AuroraSmallPretrained` with `strict=True`, and re-saved as safetensors.
* `aurora-0.25-static.pickle` is a plain pickle of three `numpy` arrays (land-sea mask, surface
  geopotential, soil type at 0.25°) referencing only `numpy…_frombuffer` and `numpy.dtype`; it is
  loaded with a `pickle.Unpickler` whose `find_class` allows exactly those two names and re-saved as
  safetensors.

Both converted files have pinned digests, and `from_pretrained` loads only them. Fidelity: the
converted model reproduces the upstream package's own regression output for the small model within
the upstream tolerances (docs/WEIGHTS.md).

Everything model-related is imported lazily so that snapshot verification, the pickle audits and input
validation run (and can refuse) before `torch` or `aurora` are imported (fleet RTM-001). `numpy` is
used for gridded data and is imported freely.
"""

from __future__ import annotations

import copy
import hashlib
import io
import json
import math
import pickle
import pickletools
import time
import warnings
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

MODEL_ID = "microsoft/aurora"
MODEL_REVISION = "a96afd7ee6d65e3bd2d476f3be798a25a56f2296"
MODEL_LICENSE = "mit"
MODEL_KEY = "aurora-0.25-small"
ARTIFACT_FORMAT = "org.valcorza.aurora-earth-system.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source assets (both pickles; see docs/WEIGHTS.md).
SOURCE_CKPT_NAME = "aurora-0.25-small-pretrained.ckpt"
SOURCE_CKPT_BYTES = 451_339_106
SOURCE_CKPT_SHA256 = "f80f78de1524a9faba8c9053e4a8ce6a2114ec01cff7f7b4efe9377200d50621"
SOURCE_STATIC_NAME = "aurora-0.25-static.pickle"
SOURCE_STATIC_BYTES = 12_459_115
SOURCE_STATIC_SHA256 = "e382103f6b24bcf1f996cc0af217c71ff2fc66507a5221e1300b5017581bd318"
# Code-free serving files produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_WEIGHTS_NAME = "aurora-0.25-small-pretrained.safetensors"
CONVERTED_STATIC_NAME = "aurora-0.25-static.safetensors"
CONVERTED_SHA256 = {
    CONVERTED_WEIGHTS_NAME: "fc03b5fc5764e08e1f7a05f06ec4debb87fbb66fa2496ec203142221d0843370",
    CONVERTED_STATIC_NAME: "9bd430b666d9267aca34d5c7924d594c3474296c9e39d85701df389816303bc0",
}
CONVERTED_BYTES = {CONVERTED_WEIGHTS_NAME: 451_230_408, CONVERTED_STATIC_NAME: 12_459_136}
# Static-audit digests of the two source pickles (sorted global names), see `audit_pickle`.
PICKLE_AUDIT_SHA256 = {
    SOURCE_CKPT_NAME: "e7b998d087a5dcadd37713daf30b63cc571160c3180ebc138500ab662197e932",
    SOURCE_STATIC_NAME: "aeec283f2dbb5861afffe185d1ee13e6df5b6f8616a475b38310c0791085c27f",
}
# What each pickle may reference. The checkpoint is a state dict; the static file is three arrays.
CKPT_ALLOWED_GLOBALS = frozenset({"collections.OrderedDict", "torch._utils._rebuild_tensor_v2", "torch.FloatStorage"})
STATIC_ALLOWED_GLOBALS = frozenset({"numpy.core.numeric._frombuffer", "numpy._core.numeric._frombuffer", "numpy.dtype"})

# Architecture and data-contract facts.
MODEL_CLASS = "AuroraSmallPretrained"
PARAMETER_COUNT = 112_797_584
STATE_TENSORS = 332
LORA_TENSORS = 80
LORA_PARAMETERS = 540_672
SURF_VARS: tuple[str, ...] = ("2t", "10u", "10v", "msl")
ATMOS_VARS: tuple[str, ...] = ("t", "u", "v", "q", "z")
STATIC_VARS: tuple[str, ...] = ("lsm", "z", "slt")
LEVELS: tuple[int, ...] = (50, 100, 150, 200, 250, 300, 400, 500, 600, 700, 850, 925, 1000)
TIMESTEP_HOURS = 6
HISTORY_STEPS = 2
PATCH_SIZE = 4
NATIVE_SHAPE = (721, 1440)
MIN_GRID = (17, 32)
MAX_GRID = NATIVE_SHAPE
MAX_STEPS_PER_WINDOW = 64
MAX_ROLLOUT_STEPS = 8
UNITS = {"2t": "K", "10u": "m/s", "10v": "m/s", "msl": "Pa", "t": "K", "u": "m/s", "v": "m/s", "q": "kg/kg", "z": "m²/s²"}
# Physical plausibility ranges (refusal only; nothing checks meteorological consistency).
RANGES = {
    "2t": (150.0, 350.0),
    "10u": (-150.0, 150.0),
    "10v": (-150.0, 150.0),
    "msl": (85_000.0, 110_000.0),
    "t": (150.0, 350.0),
    "u": (-300.0, 300.0),
    "v": (-300.0, 300.0),
    "q": (-1e-3, 0.1),
    "z": (-10_000.0, 250_000.0),
    "lsm": (0.0, 1.0),
    "slt": (0.0, 10.0),
}
LOSS_SCALES = {"2t": 10.0, "10u": 5.0, "10v": 5.0, "msl": 1000.0, "t": 10.0, "u": 15.0, "v": 15.0, "q": 4e-3, "z": 25_000.0}
_ISO = "%Y-%m-%dT%H:%M:%S"


# --------------------------------------------------------------------------------------------------
# manifest, staging, static pickle audits and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    for required in (SOURCE_CKPT_NAME, SOURCE_STATIC_NAME):
        if required not in listed:
            raise ValueError(f"manifest does not list {required}; refusing to proceed")
    pinned = {
        SOURCE_CKPT_NAME: (SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256),
        SOURCE_STATIC_NAME: (SOURCE_STATIC_BYTES, SOURCE_STATIC_SHA256),
    }
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] in pinned and (size, digest) != pinned[entry["path"]]:
            raise ValueError(f"{entry['path']}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the converted serving files (weights + static fields, safetensors) against the pinned digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    report: dict[str, Any] = {"files": []}
    for name, expected in CONVERTED_SHA256.items():
        file_path = root / name
        if not file_path.is_file():
            raise FileNotFoundError(f"converted file missing: {file_path}")
        size = file_path.stat().st_size
        if size != CONVERTED_BYTES[name]:
            raise ValueError(f"{name}: size {size} != pinned {CONVERTED_BYTES[name]}")
        digest = _sha256_file(file_path)
        if digest != expected:
            raise ValueError(f"{name}: sha256 {digest} != pinned {expected}")
        report["files"].append({"path": name, "bytes": size, "sha256": digest})
    return report


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and, when
    the converted serving files are present, those against the pinned digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = all((root / name).is_file() for name in CONVERTED_SHA256)
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and git-ignores
    the 451 MB checkpoint, the 12 MB static pickle and the safetensors they convert to)."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _pickle_globals(data: bytes) -> dict[str, int]:
    """Every global a pickle stream would import, collected with `pickletools.genops` (no execution)."""
    found: dict[str, int] = {}
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            found[key] = found.get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            found[key] = found.get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)
    return found


def audit_pickle(path: str | Path, *, allowed: frozenset[str]) -> dict[str, Any]:
    """Statically list the globals a pickle (plain, or inside a torch zip archive) would import and refuse
    any outside `allowed`. Executes nothing. Returns the sorted globals and their digest."""
    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"file not found: {file_path}")
    data = file_path.read_bytes()
    found: dict[str, int] = {}
    nested = 0
    if data[:4] == b"PK\x03\x04":
        archive = zipfile.ZipFile(io.BytesIO(data))
        for name in archive.namelist():
            if name.endswith(".pkl"):
                nested += 1
                for key, count in _pickle_globals(archive.read(name)).items():
                    found[key] = found.get(key, 0) + count
    else:
        found = _pickle_globals(data)
    violations = sorted(name for name in found if name not in allowed)
    summary = {
        "file": file_path.name,
        "torch_archive": data[:4] == b"PK\x03\x04",
        "pickles": nested if nested else 1,
        "globals": sorted(found),
        "violations": violations,
        "audit_sha256": hashlib.sha256("\n".join(sorted(found)).encode("utf-8")).hexdigest(),
    }
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed, globals outside the allow-list: {violations}")
    return summary


class _RestrictedUnpickler(pickle.Unpickler):
    """`find_class` limited to an exact allow-list of `module.name` strings."""

    def __init__(self, stream: Any, allowed: frozenset[str]) -> None:
        super().__init__(stream)
        self._allowed = allowed

    def find_class(self, module: str, name: str) -> Any:
        if f"{module}.{name}" not in self._allowed:
            raise pickle.UnpicklingError(f"refused global {module}.{name}")
        return super().find_class(module, name)


def _check_pinned_source(
    root: Path, name: str, size_expected: int, digest_expected: str, allowed: frozenset[str]
) -> dict[str, Any]:
    source = root / name
    if not source.is_file():
        raise FileNotFoundError(f"source file not found: {source}")
    size = source.stat().st_size
    if size != size_expected:
        raise ValueError(f"{name}: size {size} != pinned {size_expected}")
    digest = _sha256_file(source)
    if digest != digest_expected:
        raise ValueError(f"{name}: sha256 {digest} != pinned {digest_expected}")
    audit = audit_pickle(source, allowed=allowed)
    if audit["audit_sha256"] != PICKLE_AUDIT_SHA256[name]:
        raise ValueError(f"{name}: pickle audit digest {audit['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256[name]}")
    return {"path": name, "bytes": size, "sha256": digest, "audit": audit}


def build_model(*, use_lora: bool = False) -> Any:
    """Instantiate the small pretrained architecture from the installed `microsoft-aurora` package."""
    from aurora import AuroraSmallPretrained

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return AuroraSmallPretrained(use_lora=use_lora)


def convert_model(path: str | Path | None = None) -> dict[str, Any]:
    """Convert both pinned pickles into safetensors, deterministically, after size, digest and static-audit
    checks. The checkpoint goes through torch's weights-only unpickler and the upstream compatibility shim
    into a strictly loaded model whose state dict is saved; the static fields go through a `find_class`
    allow-list of two numpy names. Returns both identities."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    ckpt = _check_pinned_source(root, SOURCE_CKPT_NAME, SOURCE_CKPT_BYTES, SOURCE_CKPT_SHA256, CKPT_ALLOWED_GLOBALS)
    static = _check_pinned_source(root, SOURCE_STATIC_NAME, SOURCE_STATIC_BYTES, SOURCE_STATIC_SHA256, STATIC_ALLOWED_GLOBALS)
    import numpy as np
    import torch
    from safetensors.torch import save_file

    started = time.perf_counter()
    state = torch.load(root / SOURCE_CKPT_NAME, map_location="cpu", weights_only=True)
    if not isinstance(state, dict) or any(not isinstance(v, torch.Tensor) for v in state.values()):
        raise ValueError(f"{SOURCE_CKPT_NAME} did not unpickle to a state dict of tensors")
    model = build_model(use_lora=False)
    adapted = model._adapt_checkpoint(dict(state))
    model.load_state_dict(adapted, strict=True)
    canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
    if len(canonical) != STATE_TENSORS or sum(v.numel() for v in canonical.values()) != PARAMETER_COUNT:
        raise ValueError(
            f"converted state dict has {len(canonical)} tensors; expected {STATE_TENSORS} with {PARAMETER_COUNT} parameters"
        )
    save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})

    with open(root / SOURCE_STATIC_NAME, "rb") as fh:
        fields = _RestrictedUnpickler(fh, STATIC_ALLOWED_GLOBALS).load()
    if not isinstance(fields, dict) or set(fields) != set(STATIC_VARS):
        raise ValueError(f"{SOURCE_STATIC_NAME} did not unpickle to the three static fields {STATIC_VARS}")
    tensors = {}
    for name in STATIC_VARS:
        array = np.asarray(fields[name], dtype=np.float32)
        if array.shape != NATIVE_SHAPE or not np.all(np.isfinite(array)):
            raise ValueError(f"{SOURCE_STATIC_NAME}: field {name} has shape {array.shape} or non-finite values")
        tensors[name] = torch.from_numpy(np.ascontiguousarray(array))
    save_file(tensors, str(root / CONVERTED_STATIC_NAME), metadata={"format": "pt"})
    report = verify_converted(root)
    return {
        "sources": [{k: v for k, v in ckpt.items() if k != "audit"}, {k: v for k, v in static.items() if k != "audit"}],
        "audits": {SOURCE_CKPT_NAME: ckpt["audit"], SOURCE_STATIC_NAME: static["audit"]},
        "converted": report["files"],
        "seconds": round(time.perf_counter() - started, 2),
    }


# --------------------------------------------------------------------------------------------------
# gridded windows and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "one window: {lat, lon, levels, times, surf: {2t, 10u, 10v, msl: (T, H, W)}, atmos: {t, u, v, q, z: (T, 13, H, W)}, "
        "static: {lsm, z, slt: (H, W)}} on a global equiangular grid"
    ),
    "grid": (
        f"H in {MIN_GRID[0]}..{MAX_GRID[0]} latitudes (90 → -90, with poles; H % 4 in (0, 1)), "
        f"W in {MIN_GRID[1]}..{MAX_GRID[1]} longitudes (0 → 360, W % 4 == 0)"
    ),
    "levels": list(LEVELS),
    "time_steps": [HISTORY_STEPS + 1, MAX_STEPS_PER_WINDOW],
    "timestep_hours": TIMESTEP_HOURS,
    "history_steps": HISTORY_STEPS,
    "rollout_steps": [1, MAX_ROLLOUT_STEPS],
    "units": dict(UNITS),
    "ranges": {k: list(v) for k, v in RANGES.items()},
    "validation": (
        "variable names, array shapes, grid monotonicity and spacing, 6-hour time spacing, finiteness and physical "
        "ranges only. Nothing checks that the fields are dynamically consistent, that the analysis is real, or that "
        "the resolution is one the model was trained on -- a smooth random field within range is forecast without complaint"
    ),
}


def _parse_time(value: Any) -> datetime:
    if isinstance(value, datetime):
        return value.replace(tzinfo=None)
    if isinstance(value, str):
        try:
            return datetime.strptime(value[:19], _ISO)
        except ValueError as exc:
            raise ValueError(f"time {value!r} is not ISO 8601 (YYYY-MM-DDTHH:MM:SS)") from exc
    raise ValueError(f"time {value!r} must be a datetime or an ISO 8601 string")


def _check_window(window: Mapping[str, Any]) -> dict[str, Any]:
    """Validate one window and return a normalised copy (float32 arrays, lat 90→-90, lon 0→360)."""
    import numpy as np

    if not isinstance(window, Mapping):
        raise ValueError("window must be a mapping with lat/lon/levels/times/surf/atmos/static")
    for key in ("lat", "lon", "levels", "times", "surf", "atmos", "static"):
        if key not in window:
            raise ValueError(f"window is missing {key!r}")
    lat = np.asarray(window["lat"], dtype=np.float64)
    lon = np.asarray(window["lon"], dtype=np.float64)
    if lat.ndim != 1 or lon.ndim != 1 or not np.all(np.isfinite(lat)) or not np.all(np.isfinite(lon)):
        raise ValueError("lat and lon must be finite 1-D arrays")
    height, width = len(lat), len(lon)
    if not MIN_GRID[0] <= height <= MAX_GRID[0] or not MIN_GRID[1] <= width <= MAX_GRID[1]:
        raise ValueError(f"grid {height}x{width} is outside {MIN_GRID}..{MAX_GRID}")
    if width % PATCH_SIZE:
        raise ValueError(f"the number of longitudes ({width}) must be a multiple of the patch size {PATCH_SIZE}")
    if height % PATCH_SIZE not in (0, 1):
        raise ValueError(f"the number of latitudes ({height}) must be a multiple of {PATCH_SIZE}, or one more")
    flip = False
    dlat = np.diff(lat)
    if np.all(dlat > 0):
        flip = True
        lat = lat[::-1]
        dlat = -dlat[::-1]
    if not np.all(dlat < 0) or not np.allclose(dlat, dlat[0], atol=1e-6):
        raise ValueError("lat must be strictly monotonic with uniform spacing")
    if not (abs(lat[0] - 90.0) < 1e-6 and abs(lat[-1] + 90.0) < 1e-6):
        raise ValueError("lat must span the poles: 90 to -90 (Aurora is a global model)")
    dlon = np.diff(lon)
    if not np.all(dlon > 0) or not np.allclose(dlon, dlon[0], atol=1e-6) or lon[0] < 0.0 or lon[-1] >= 360.0:
        raise ValueError("lon must be strictly increasing with uniform spacing within [0, 360)")
    if not np.isclose(lon[0] + 360.0 - lon[-1], dlon[0], atol=1e-6):
        raise ValueError("lon must cover the full circle (last step wraps to the first)")
    levels = [int(x) for x in np.asarray(window["levels"]).tolist()]
    if tuple(levels) != LEVELS:
        raise ValueError(f"levels must be exactly {LEVELS}, got {tuple(levels)}")
    times = [_parse_time(t) for t in window["times"]]
    n_steps = len(times)
    if not HISTORY_STEPS + 1 <= n_steps <= MAX_STEPS_PER_WINDOW:
        raise ValueError(f"a window needs {HISTORY_STEPS + 1}..{MAX_STEPS_PER_WINDOW} time steps, got {n_steps}")
    for earlier, later in zip(times, times[1:], strict=False):
        if later - earlier != timedelta(hours=TIMESTEP_HOURS):
            raise ValueError(f"times must be spaced exactly {TIMESTEP_HOURS} h apart ({earlier} -> {later})")

    def field_array(group: str, name: str, shape: tuple[int, ...]) -> Any:
        source = window[group]
        if not isinstance(source, Mapping) or name not in source:
            raise ValueError(f"{group} is missing variable {name!r}")
        try:
            array = np.asarray(source[name], dtype=np.float32)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{group}/{name} must be a numeric array") from exc
        if array.shape != shape:
            raise ValueError(f"{group}/{name} has shape {array.shape}, expected {shape}")
        if not np.all(np.isfinite(array)):
            raise ValueError(f"{group}/{name} contains non-finite values")
        low, high = RANGES.get(name if group != "static" else name, (-math.inf, math.inf))
        if group == "static" and name == "z":
            low, high = RANGES["z"]
        if float(array.min()) < low or float(array.max()) > high:
            raise ValueError(
                f"{group}/{name} has values outside the plausible range {(low, high)} {UNITS.get(name, '')}".rstrip()
            )
        if flip:
            array = array[..., ::-1, :]
        return np.ascontiguousarray(array)

    surf = {name: field_array("surf", name, (n_steps, height, width)) for name in SURF_VARS}
    atmos = {name: field_array("atmos", name, (n_steps, len(LEVELS), height, width)) for name in ATMOS_VARS}
    static = {name: field_array("static", name, (height, width)) for name in STATIC_VARS}
    unknown = sorted(set(window["surf"]) - set(SURF_VARS)) + sorted(set(window["atmos"]) - set(ATMOS_VARS))
    name = window.get("name")
    if name is not None and (not isinstance(name, str) or len(name) > 200):
        raise ValueError("name must be a string of at most 200 characters")
    return {
        "name": name if name is not None else f"window-{times[0].strftime('%Y%m%d%H')}",
        "lat": lat.tolist(),
        "lon": lon.tolist(),
        "levels": list(LEVELS),
        "times": [t.strftime(_ISO) for t in times],
        "surf": surf,
        "atmos": atmos,
        "static": static,
        "shape": (height, width),
        "n_steps": n_steps,
        "flipped_latitude": flip,
        "ignored_variables": unknown,
    }


def window_digest(window: Mapping[str, Any]) -> str:
    """SHA-256 over the grid, times and every field (float32 bytes) of a validated window."""
    checked = _check_window(window)
    digest = hashlib.sha256()
    digest.update(json.dumps({"lat": checked["lat"], "lon": checked["lon"], "times": checked["times"]}).encode("utf-8"))
    for group in ("surf", "atmos", "static"):
        for name in sorted(checked[group]):
            digest.update(name.encode("utf-8"))
            digest.update(checked[group][name].tobytes())
    return digest.hexdigest()


def validate_inputs(window: Mapping[str, Any]) -> dict[str, Any]:
    """Structural validation only; raises ValueError before any model library is imported."""
    checked = _check_window(window)
    return {
        "name": checked["name"],
        "shape": checked["shape"],
        "n_steps": checked["n_steps"],
        "times": [checked["times"][0], checked["times"][-1]],
        "forecast_origins": checked["n_steps"] - HISTORY_STEPS,
        "flipped_latitude": checked["flipped_latitude"],
        "ignored_variables": checked["ignored_variables"],
        "resolution_degrees": round(360.0 / checked["shape"][1], 4),
        "native_resolution": checked["shape"] == NATIVE_SHAPE,
    }


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------


def _lat_weights(lat: Sequence[float]) -> Any:
    import numpy as np

    weights = np.cos(np.deg2rad(np.asarray(lat, dtype=np.float64)))
    return weights / weights.mean()


@dataclass
class AuroraPipeline:
    """6-hourly global forecasting and bounded LoRA fine-tuning on top of the verified Aurora small model."""

    model: Any
    device: str
    weights_dir: Path
    source: str
    use_lora: bool
    adapter: dict[str, Any] | None = None
    _static_native: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str = "cpu",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        use_lora: bool = False,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> AuroraPipeline:
        """Verify, convert if needed, build from the installed package and strictly load. `use_lora=True`
        adds the (zero-initialised, behaviour-preserving) LoRA parameters that `adapt` trains. With
        `require_source=False` the pickles may be absent (the DIMER-hosted case) as long as the converted
        files verify. `report` receives the audit and conversion records when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source or (root / MANIFEST_NAME).is_file():
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted files already present and digest-verified"})
            source = "converted from the manifest-verified source pickles"
        else:
            verify_converted(root)
            source = "converted files, pinned digests (source pickles absent)"
        import torch
        from safetensors.torch import load_file

        model = build_model(use_lora=use_lora)
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        result = model.load_state_dict(state, strict=not use_lora)
        if use_lora and (
            result.unexpected_keys
            or any("lora" not in k for k in result.missing_keys)
            or len(result.missing_keys) != LORA_TENSORS
        ):
            raise ValueError(
                f"unexpected state-dict layout with LoRA: missing={len(result.missing_keys)} unexpected={result.unexpected_keys}"
            )
        n_params = sum(p.numel() for n, p in model.named_parameters() if "lora" not in n)
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} base parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(device)).eval()
        for param in model.parameters():
            param.requires_grad_(False)
        static = load_file(str(root / CONVERTED_STATIC_NAME))
        return cls(model=model, device=device, weights_dir=root, source=source, use_lora=use_lora, _static_native=static)

    # ---- batches --------------------------------------------------------------------------------------

    def native_static_fields(self) -> dict[str, Any]:
        """The upstream 0.25° static fields (lsm, z, slt) as numpy arrays of shape (721, 1440), lat 90→-90."""
        return {k: v.numpy() for k, v in self._static_native.items()}

    def _batch(self, checked: Mapping[str, Any], origin: int) -> Any:
        """Model input for the forecast origin `origin` (the step index of the latest analysis)."""
        import torch
        from aurora import Batch, Metadata

        if origin < HISTORY_STEPS - 1 or origin >= checked["n_steps"]:
            raise ValueError(f"origin must be in {HISTORY_STEPS - 1}..{checked['n_steps'] - 1}")
        lo, hi = origin - HISTORY_STEPS + 1, origin + 1
        return Batch(
            surf_vars={k: torch.from_numpy(v[lo:hi][None]) for k, v in checked["surf"].items()},
            static_vars={k: torch.from_numpy(v) for k, v in checked["static"].items()},
            atmos_vars={k: torch.from_numpy(v[lo:hi][None]) for k, v in checked["atmos"].items()},
            metadata=Metadata(
                lat=torch.tensor(checked["lat"], dtype=torch.float32),
                lon=torch.tensor(checked["lon"], dtype=torch.float32),
                time=(_parse_time(checked["times"][origin]),),
                atmos_levels=tuple(LEVELS),
            ),
        )

    def _forecast(self, batch: Any, steps: int) -> list[Any]:
        """Autoregressive roll-out of `steps` × 6 h; each element is the model's Batch for that lead."""
        import torch
        from aurora import rollout

        with torch.inference_mode():
            return [pred.to("cpu") for pred in rollout(self.model, batch.to(self.device), steps=steps)]

    # ---- inference ------------------------------------------------------------------------------------

    def predict(self, window: Mapping[str, Any], *, origin: int | None = None, steps: int = 1) -> dict[str, Any]:
        """Forecast `steps` × 6 h from the analysis at `origin` (default: the last step of the window)."""
        if not isinstance(steps, int) or not 1 <= steps <= MAX_ROLLOUT_STEPS:
            raise ValueError(f"steps must be an int in 1..{MAX_ROLLOUT_STEPS}")
        checked = _check_window(window)
        if origin is None:
            origin = checked["n_steps"] - 1
        started = time.perf_counter()
        batch = self._batch(checked, origin)
        preds = self._forecast(batch, steps)
        origin_time = _parse_time(checked["times"][origin])
        forecasts = []
        for k, pred in enumerate(preds, start=1):
            forecasts.append(
                {
                    "lead_hours": k * TIMESTEP_HOURS,
                    "valid_time": (origin_time + timedelta(hours=k * TIMESTEP_HOURS)).strftime(_ISO),
                    "surf": {name: pred.surf_vars[name][0, 0].numpy() for name in SURF_VARS},
                    "atmos": {name: pred.atmos_vars[name][0, 0].numpy() for name in ATMOS_VARS},
                }
            )
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "name": checked["name"],
            "origin_time": origin_time.strftime(_ISO),
            "shape": (len(preds[0].metadata.lat), len(preds[0].metadata.lon)),
            "lat": preds[0].metadata.lat.tolist(),
            "lon": preds[0].metadata.lon.tolist(),
            "units": dict(UNITS),
            "forecasts": forecasts,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(
        self, window: Mapping[str, Any], *, max_lead_steps: int = 1, origins: Sequence[int] | None = None
    ) -> dict[str, Any]:
        """Latitude-weighted RMSE per variable and lead time against the window's later analyses, with
        the persistence forecast (the latest analysis carried forward) scored the same way."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import forecast_metrics` removed — names are kernel globals defined by the carried modules

        if not isinstance(max_lead_steps, int) or not 1 <= max_lead_steps <= MAX_ROLLOUT_STEPS:
            raise ValueError(f"max_lead_steps must be an int in 1..{MAX_ROLLOUT_STEPS}")
        checked = _check_window(window)
        last_origin = checked["n_steps"] - 1 - max_lead_steps
        if last_origin < HISTORY_STEPS - 1:
            raise ValueError(
                f"window has {checked['n_steps']} steps; {HISTORY_STEPS + max_lead_steps} are needed "
                f"for {max_lead_steps} lead steps"
            )
        chosen = list(origins) if origins is not None else list(range(HISTORY_STEPS - 1, last_origin + 1))
        for origin in chosen:
            if not HISTORY_STEPS - 1 <= origin <= last_origin:
                raise ValueError(f"origin {origin} is outside {HISTORY_STEPS - 1}..{last_origin}")
        started = time.perf_counter()
        predictions = []
        for origin in chosen:
            preds = self._forecast(self._batch(checked, origin), max_lead_steps)
            predictions.append(
                [
                    {
                        "surf": {n: p.surf_vars[n][0, 0].numpy() for n in SURF_VARS},
                        "atmos": {n: p.atmos_vars[n][0, 0].numpy() for n in ATMOS_VARS},
                    }
                    for p in preds
                ]
            )
        metrics = forecast_metrics(checked, chosen, predictions)
        metrics["seconds"] = round(time.perf_counter() - started, 3)
        return metrics

    # ---- adaptation -----------------------------------------------------------------------------------

    def _trainable(self, mode: str) -> list[str]:
        if mode not in ("lora", "lora+heads"):
            raise ValueError("trainable must be 'lora' or 'lora+heads'")
        if not self.use_lora:
            raise ValueError("adapt() needs a pipeline built with use_lora=True")
        prefixes = ("decoder.surf_heads.", "decoder.atmos_heads.", "encoder.surf_token_embeds.", "encoder.atmos_token_embeds.")
        names = []
        for name, _param in self.model.named_parameters():
            if "lora" in name or (mode == "lora+heads" and name.startswith(prefixes)):
                names.append(name)
        return names

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Mapping[str, Any] | None = None,
        *,
        epochs: int = 6,
        lr: float = 1e-3,
        trainable: str = "lora",
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning on one-step (6 h) forecasts from every origin of every training window.

        `trainable="lora"` trains the 80 LoRA tensors (540,672 parameters) the upstream architecture
        provides in every backbone attention block, zero-initialised so epoch 0 is the pretrained model;
        `"lora+heads"` also unfreezes the encoder token embeddings and decoder heads. Loss = mean over the
        nine variables of MSE / scale², AdamW, fixed learning rate, one origin per step. The epoch with the
        lowest validation loss is kept; epoch 0 records the frozen model."""
        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 0.1):
            raise ValueError("lr must be in (0, 0.1]")
        if not train:
            raise ValueError("at least one training window is required")
        names = self._trainable(trainable)
        train_checked = [_check_window(w) for w in train]
        val_checked = _check_window(val) if val is not None else None
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        for name, param in model.named_parameters():
            param.requires_grad_(name in set(names))
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        optimiser = torch.optim.AdamW(params, lr=lr)
        scales = {k: torch.tensor(v, dtype=torch.float32, device=self.device) for k, v in LOSS_SCALES.items()}

        def targets(checked: Mapping[str, Any], origin: int) -> dict[str, Any]:
            height = checked["shape"][0] - (checked["shape"][0] % PATCH_SIZE)
            return {
                **{k: torch.from_numpy(v[origin + 1, :height]).to(self.device) for k, v in checked["surf"].items()},
                **{k: torch.from_numpy(v[origin + 1, :, :height]).to(self.device) for k, v in checked["atmos"].items()},
            }

        def loss_of(pred: Any, target: Mapping[str, Any]) -> Any:
            total = 0.0
            for k in SURF_VARS:
                total = total + torch.mean(((pred.surf_vars[k][0, 0] - target[k]) / scales[k]) ** 2)
            for k in ATMOS_VARS:
                total = total + torch.mean(((pred.atmos_vars[k][0, 0] - target[k]) / scales[k]) ** 2)
            return total / (len(SURF_VARS) + len(ATMOS_VARS))

        def val_loss() -> float | None:
            if val_checked is None:
                return None
            model.eval()
            losses = []
            with torch.inference_mode():
                for origin in range(HISTORY_STEPS - 1, val_checked["n_steps"] - 1):
                    pred = model.forward(self._batch(val_checked, origin).to(self.device))
                    losses.append(float(loss_of(pred, targets(val_checked, origin))))
            return sum(losses) / len(losses)

        history: list[dict[str, Any]] = []
        best_state = copy.deepcopy({k: v.detach().clone() for k, v in model.state_dict().items() if k in set(names)})
        best_epoch = 0
        entry: dict[str, Any] = {
            "epoch": 0,
            "train_loss": None,
            "val_loss": val_loss(),
            "note": "frozen model (LoRA zero-initialised)",
        }
        if val_checked is not None:
            entry["val"] = self.evaluate(val_checked)["variables"]
        history.append(entry)
        best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
        if progress:
            progress(entry)
        samples = [(w, origin) for w in train_checked for origin in range(HISTORY_STEPS - 1, w["n_steps"] - 1)]
        generator = torch.Generator().manual_seed(seed)
        for epoch in range(1, epochs + 1):
            model.train()
            order = torch.randperm(len(samples), generator=generator).tolist()
            losses = []
            for index in order:
                checked, origin = samples[index]
                pred = model.forward(self._batch(checked, origin).to(self.device))
                loss = loss_of(pred, targets(checked, origin))
                optimiser.zero_grad(set_to_none=True)
                loss.backward()
                optimiser.step()
                losses.append(float(loss.detach()))
            model.eval()
            entry = {"epoch": epoch, "train_loss": sum(losses) / len(losses), "val_loss": val_loss()}
            if val_checked is not None:
                entry["val"] = self.evaluate(val_checked)["variables"]
            history.append(entry)
            if progress:
                progress(entry)
            if entry["val_loss"] is None or entry["val_loss"] < best_val:
                best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                best_state = {k: v.detach().clone() for k, v in model.state_dict().items() if k in set(names)}
                best_epoch = epoch
        merged = dict(model.state_dict())
        merged.update(best_state)
        model.load_state_dict(merged, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable": trainable,
            "trainable_names": names,
            "n_trainable": n_trainable,
            "n_total": sum(p.numel() for p in model.parameters()),
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "loss_scales": dict(LOSS_SCALES),
            "n_train_windows": len(train_checked),
            "n_train_samples": len(samples),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors (LoRA, plus heads when trained) as safetensors with a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        names = set(self.adapter["trainable_names"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k in names}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "converted_sha256": dict(CONVERTED_SHA256),
            },
            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest and digest, then overwrite exactly the tensors it carries."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records different converted-base digests")
        entry = manifest["files"][0]
        weights_path = root / entry["path"]
        digest = _sha256_file(weights_path)
        if digest != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        if any("lora" in name for name in manifest["tensors"]) and not self.use_lora:
            raise ValueError("this adapter carries LoRA tensors; build the pipeline with use_lora=True")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != manifest["tensors"]:
            raise ValueError("artifact tensor names differ from its manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if key not in state:
                raise ValueError(f"artifact tensor {key} is not part of the model")
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, model has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "trainable_names": manifest["tensors"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str = "cpu",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> AuroraPipeline:
        manifest = json.loads((Path(artifact_dir) / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        use_lora = any("lora" in name for name in manifest.get("tensors", []))
        pipeline = cls.from_pretrained(
            device=device,
            weights_dir=weights_dir,
            allow_download=allow_download,
            require_source=require_source,
            use_lora=use_lora,
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 2/3:** `src/aurora_earth_system_pipeline/metrics.py` (carried verbatim; see the note above)

In [ ]:
"""Forecast verification: latitude-weighted RMSE per variable and lead time, the persistence baseline,
and a summary skill ratio.

RMSE is weighted by cos(latitude) normalised to unit mean, the WeatherBench convention, so the poles do
not dominate an equiangular grid. Atmospheric variables are averaged over all 13 pressure levels, and
geopotential at 500 hPa (`z500`) is reported on its own because it is the customary headline. The
**persistence** forecast carries the latest analysis forward unchanged; at 6 h it is a strong baseline
that any forecast model has to beat, and it decays with lead time. The skill ratio is
RMSE(model) / RMSE(persistence): below 1 the model beats persistence.
"""

from __future__ import annotations

import math
from collections.abc import Mapping, Sequence
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import ATMOS_VARS, LEVELS, PATCH_SIZE, SURF_VARS, TIMESTEP_HOURS, _lat_weights` removed — names are kernel globals defined by the carried modules

HEADLINE_LEVEL = 500
REPORTED = (*SURF_VARS, *ATMOS_VARS, "z500")


def lat_weighted_rmse(pred: Any, truth: Any, lat: Sequence[float]) -> float:
    """Latitude-weighted RMSE over (H, W) or (L, H, W) arrays; `lat` matches the H axis."""
    import numpy as np

    pred = np.asarray(pred, dtype=np.float64)
    truth = np.asarray(truth, dtype=np.float64)
    if pred.shape != truth.shape:
        raise ValueError(f"prediction shape {pred.shape} != truth shape {truth.shape}")
    weights = _lat_weights(lat)
    weights = weights[:, None] if pred.ndim == 2 else weights[None, :, None]
    return float(np.sqrt(np.mean((pred - truth) ** 2 * weights)))


def forecast_metrics(
    window: Mapping[str, Any], origins: Sequence[int], predictions: Sequence[Sequence[Mapping[str, Any]]]
) -> dict[str, Any]:
    """Score roll-outs from `origins` of a validated window. `predictions[i][k]` is the forecast from
    `origins[i]` at lead step k+1 with `surf` / `atmos` numpy fields (H', W) where H' = H rounded down to
    the patch size, the rows the model actually predicts."""
    if len(origins) != len(predictions) or not origins:
        raise ValueError("origins and predictions must be non-empty and equal in length")
    height = window["shape"][0] - (window["shape"][0] % PATCH_SIZE)
    lat = window["lat"][:height]
    n_leads = len(predictions[0])
    level_index = LEVELS.index(HEADLINE_LEVEL)
    per_lead: dict[str, dict[str, dict[str, float]]] = {name: {} for name in REPORTED}
    for k in range(n_leads):
        lead = f"{(k + 1) * TIMESTEP_HOURS}h"
        errors: dict[str, list[float]] = {name: [] for name in REPORTED}
        persist: dict[str, list[float]] = {name: [] for name in REPORTED}
        for origin, preds in zip(origins, predictions, strict=True):
            if len(preds) != n_leads:
                raise ValueError("every origin must carry the same number of lead steps")
            target = origin + k + 1
            if target >= window["n_steps"]:
                raise ValueError(f"origin {origin} has no analysis at lead step {k + 1}")
            for name in SURF_VARS:
                truth = window["surf"][name][target, :height]
                errors[name].append(lat_weighted_rmse(preds[k]["surf"][name], truth, lat))
                persist[name].append(lat_weighted_rmse(window["surf"][name][origin, :height], truth, lat))
            for name in ATMOS_VARS:
                truth = window["atmos"][name][target, :, :height]
                errors[name].append(lat_weighted_rmse(preds[k]["atmos"][name], truth, lat))
                persist[name].append(lat_weighted_rmse(window["atmos"][name][origin, :, :height], truth, lat))
            truth = window["atmos"]["z"][target, level_index, :height]
            errors["z500"].append(lat_weighted_rmse(preds[k]["atmos"]["z"][level_index], truth, lat))
            persist["z500"].append(lat_weighted_rmse(window["atmos"]["z"][origin, level_index, :height], truth, lat))
        for name in REPORTED:
            model = sum(errors[name]) / len(errors[name])
            baseline = sum(persist[name]) / len(persist[name])
            per_lead[name][lead] = {
                "model": model,
                "persistence": baseline,
                "skill": model / baseline if baseline > 0 else math.nan,
            }
    leads = [f"{(k + 1) * TIMESTEP_HOURS}h" for k in range(n_leads)]
    summary = {
        lead: {
            "mean_skill": sum(per_lead[name][lead]["skill"] for name in REPORTED if name != "z500") / (len(REPORTED) - 1),
            "variables_beating_persistence": sum(
                1 for name in REPORTED if name != "z500" and per_lead[name][lead]["skill"] < 1.0
            ),
        }
        for lead in leads
    }
    return {
        "n_origins": len(origins),
        "leads": leads,
        "grid": [height, window["shape"][1]],
        "variables": per_lead,
        "summary": summary,
        "metric": "latitude-weighted RMSE (cos-lat weights, unit mean); atmospheric variables averaged over the 13 levels",
        "units": "as the variables: K, m/s, Pa, kg/kg, m²/s²",
    }


def persistence_only(window: Mapping[str, Any], *, max_lead_steps: int = 1) -> dict[str, Any]:
    """The persistence baseline alone (no model): the same table with the model column omitted."""
    import numpy as np

    pass  # standalone rewrite (build_notebook.py): `from .pipeline import HISTORY_STEPS, _check_window` removed — names are kernel globals defined by the carried modules

    checked = _check_window(window)
    last_origin = checked["n_steps"] - 1 - max_lead_steps
    origins = list(range(HISTORY_STEPS - 1, last_origin + 1))
    if not origins:
        raise ValueError("window too short for the requested lead")
    height = checked["shape"][0] - (checked["shape"][0] % PATCH_SIZE)
    predictions = [
        [
            {
                "surf": {n: np.asarray(checked["surf"][n][origin, :height]) for n in SURF_VARS},
                "atmos": {n: np.asarray(checked["atmos"][n][origin, :, :height]) for n in ATMOS_VARS},
            }
            for _ in range(max_lead_steps)
        ]
        for origin in origins
    ]
    metrics = forecast_metrics(checked, origins, predictions)
    metrics["variables"] = {
        name: {lead: {"persistence": v["persistence"]} for lead, v in leads.items()}
        for name, leads in metrics["variables"].items()
    }
    metrics.pop("summary")
    metrics["baseline"] = "persistence (latest analysis carried forward)"
    return metrics

**Module 3/3:** `src/aurora_earth_system_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Tutorial dataset, validation, extended-NetCDF I/O for the Aurora pipeline.

The default dataset is **real ERA5 reanalysis** at 1.5° (240 × 121, with poles), served by WeatherBench 2
(Rasp et al., 2024) from a public Google Cloud Storage bucket as a Zarr v2 store. Four two-day windows
(eight 6-hourly analyses each) are fetched: two for training (January and July 2019), one for validation
(April 2020) and one for testing (October 2021) — distinct seasons and years, so nothing in the test
window is a near-duplicate of anything trained on. Every object fetched (chunk, `.zarray`, coordinate)
has its byte size and SHA-256 pinned in `WB2_OBJECTS`; a mismatch is refused before decoding, and the
Zarr chunks are decoded with `numcodecs` (Blosc/LZ4) directly, without a Zarr or xarray dependency on
the download path. About 196 MB is transferred.

Why 1.5° and not the model's native 0.25°: a native-resolution window is 60× larger (about 430 MB per
atmospheric variable and window), far beyond a tutorial's budget, and the small checkpoint is the one
upstream publishes for testing. Running the 0.25° model on a 1.5° grid is therefore an out-of-distribution
use, and the tutorial says so: the frozen model's error is measured honestly against persistence, and the
adaptation contract is what moves it. ERA5 is produced by ECMWF/Copernicus (licence to use Copernicus
products, attribution required); WeatherBench 2 redistributes it unchanged apart from regridding.
"""

from __future__ import annotations

import hashlib
import json
import urllib.request
from collections.abc import Mapping, Sequence
from datetime import datetime, timedelta
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import ATMOS_VARS, HISTORY_STEPS, STATIC_VARS, SURF_VARS, TIMESTEP_HOURS, _check_window, window_digest` removed — names are kernel globals defined by the carried modules

WB2_BASE_URL = (
    "https://storage.googleapis.com/weatherbench2/datasets/era5/"
    "1959-2023_01_10-6h-240x121_equiangular_with_poles_conservative.zarr"
)
WB2_EPOCH = datetime(1959, 1, 1)
WB2_CHUNK_STEPS = 8
WB2_SURF = {
    "2t": "2m_temperature",
    "10u": "10m_u_component_of_wind",
    "10v": "10m_v_component_of_wind",
    "msl": "mean_sea_level_pressure",
}
WB2_ATMOS = {
    "t": "temperature",
    "u": "u_component_of_wind",
    "v": "v_component_of_wind",
    "q": "specific_humidity",
    "z": "geopotential",
}
WB2_STATIC = {"lsm": "land_sea_mask", "z": "geopotential_at_surface", "slt": "soil_type"}
# Chunk index -> role. Chunk k holds analyses 8k .. 8k+7 (6-hourly from 1959-01-01 00:00).
SAMPLE_WINDOWS: dict[str, int] = {"train-2019-01": 10957, "train-2019-07": 11048, "val-2020-04": 11185, "test-2021-10": 11459}
SAMPLE_LABEL_SOURCE = "ERA5 via WeatherBench 2 (1.5°, GCS bucket weatherbench2)"
DEFAULT_CACHE_DIR = Path("weights") / "wb2-era5-1p5deg"
MIN_WINDOWS = 1

WB2_OBJECTS: dict[str, tuple[int, str]] = {
    "latitude/.zarray": (317, "1576147d9e73e403a5558d7030613586fe94e0ca1c7c9e9e14fb393b0724c8aa"),
    "longitude/.zarray": (317, "eb41dc00d27b72577623ec2ddccb7693006f14cdc9f06a81f2a93b94fa9dd10e"),
    "level/.zarray": (314, "8217857a6c10d13b295ab6ffd491092a75ac976d5af0a4e8d8f5964a61c56f84"),
    "10m_u_component_of_wind/.zarray": (369, "6372c239b61980a3807f4f7138539f9cf97e5b08eed8ab5d2f6b357a79827907"),
    "10m_u_component_of_wind/10957.0.0": (817882, "1eaf8efe88fe1c2eb8f4f7fae0c23dc587312ff1255ffe04af6afd259aacffc0"),
    "10m_u_component_of_wind/11048.0.0": (817202, "ec90cb193bdbcd316c9ed37ae70cf9e680129fb361a9ac947ff78876d6eb8bb5"),
    "10m_u_component_of_wind/11185.0.0": (813825, "3aee927c30dc6101a09cfc7f970d9edca936cf1c522033d1c89cb374cbf49fbd"),
    "10m_u_component_of_wind/11459.0.0": (817758, "16e05fa2291d35775036e2f2feebfc7541dbaf56ac96562218460e82f9c62069"),
    "10m_v_component_of_wind/.zarray": (369, "6372c239b61980a3807f4f7138539f9cf97e5b08eed8ab5d2f6b357a79827907"),
    "10m_v_component_of_wind/10957.0.0": (814637, "6bd14583e1f00014aeca0324c006860c66a994557595fa27d5d44aef8b03fdc5"),
    "10m_v_component_of_wind/11048.0.0": (818216, "ec5e60a9db34773b23b1a5a79bf9ff66148c10016febddbecb02e2cf2a29f8d5"),
    "10m_v_component_of_wind/11185.0.0": (819793, "0ed0783d4abdcdc85ce456f38700761af89f7b3f1a93b31fe537f140b1f2b365"),
    "10m_v_component_of_wind/11459.0.0": (817993, "efe882d2300adada80b870cb89c540690f6f64c92705deabfb084cc11ed95680"),
    "2m_temperature/.zarray": (369, "6372c239b61980a3807f4f7138539f9cf97e5b08eed8ab5d2f6b357a79827907"),
    "2m_temperature/10957.0.0": (615801, "13dd6303a537adec10cf5bd9446e2f595c9b62a1c26f586233978c647173a0db"),
    "2m_temperature/11048.0.0": (611125, "3e139f475ccb6f905344723e961c0a15c9fc037d67c21a511804300de4b50917"),
    "2m_temperature/11185.0.0": (616959, "3d0b090fe11d580a5b8a7dc9b6573505bddf3a6705b4d65f34ebec467300a354"),
    "2m_temperature/11459.0.0": (609456, "9fdffb77031f8372544c68e43d9431b3c26b86b0396ce13ff91a552d6c9bfb38"),
    "geopotential/.zarray": (393, "61cfdce172da056ef17cf80bd9e30d89ddff268f848813b21b488df8e8da270d"),
    "geopotential/10957.0.0.0": (7685214, "1e978f48ee0d2808ad357d2ba6d82804b5b3e20867c874e3ec1223150da33f8a"),
    "geopotential/11048.0.0.0": (7651568, "8136af69c379816caca6986615d52abfc983d6f06821f8555d12c5f61ca0b877"),
    "geopotential/11185.0.0.0": (7683052, "c6d241d6c2d04949b1cd0333634f934f82ad51d0fbe2d569e5ea1f11cd62c3d2"),
    "geopotential/11459.0.0.0": (7670669, "b55a98729b3b4ad906c4d272a82e509f850152449d882586e98c8290d75d1ecc"),
    "geopotential_at_surface/.zarray": (343, "757b13432c66096257f584133a71e5308134547aef9dbc6505ef553b2d2e8d67"),
    "geopotential_at_surface/0.0": (107690, "2df9d20c03bceb688bd1522a7c5fb154c15e01b4375c9dfe3f7fdb5348edbcbe"),
    "land_sea_mask/.zarray": (343, "757b13432c66096257f584133a71e5308134547aef9dbc6505ef553b2d2e8d67"),
    "land_sea_mask/0.0": (52875, "dcd229c9d152d5cdf422defe2bb1d2c425cfc52832078cb1b0600875f9d9a5ef"),
    "latitude/0": (528, "0099a8a0cfec2eb3e7d3c1a5a0b00b063c86ba0fe98bdc379bfe269af0207875"),
    "level/0": (120, "defe6a82a653349d9bdc64bbb6cf092e3d680d913a7f5ae7a584289df37edb79"),
    "longitude/0": (845, "127c574174be6960bd1abe3d7f6259b9b9a85fad751df91ae1e6c0c469bb0beb"),
    "mean_sea_level_pressure/.zarray": (369, "6372c239b61980a3807f4f7138539f9cf97e5b08eed8ab5d2f6b357a79827907"),
    "mean_sea_level_pressure/10957.0.0": (553842, "23530ea6b4aa7a081b6f344fce9e5aab388f695ce485ada5af172807a69231b3"),
    "mean_sea_level_pressure/11048.0.0": (555456, "727cc3a840d305eb4cc4da60d0c8cadad0634358394d21c10259c26dbf0e20dd"),
    "mean_sea_level_pressure/11185.0.0": (555386, "2c1c4870239f435dd7d053a333059167b7e41bc6a481e6f52c61ad6bbea1df62"),
    "mean_sea_level_pressure/11459.0.0": (554365, "5ce8ae2cb07ee901da9784060a533946c6ae3ce3f80030c02e0789e2f596dabd"),
    "soil_type/.zarray": (343, "757b13432c66096257f584133a71e5308134547aef9dbc6505ef553b2d2e8d67"),
    "soil_type/0.0": (47405, "7300007695fff5062fa5981260d5253cc068a1e5b6f3eab40ae08bd270b2aaeb"),
    "specific_humidity/.zarray": (393, "61cfdce172da056ef17cf80bd9e30d89ddff268f848813b21b488df8e8da270d"),
    "specific_humidity/10957.0.0.0": (9745439, "72b787c6bfbc5da541e1d0e0792365df85d99878d930e1042a068a21cd27963d"),
    "specific_humidity/11048.0.0.0": (9803016, "dacee098673741378a78ad021b093e2138e200c6caf7d312d51f3faa80b2c575"),
    "specific_humidity/11185.0.0.0": (9704312, "7eee37cd5002da29d49b52b8237c0440415a219b8bb729d7e07714477e82bfdf"),
    "specific_humidity/11459.0.0.0": (9832381, "9e73a10c98d9da9cc50be7c5bb89e93159bdcdbe989bb1ea98880c55e97cf17c"),
    "temperature/.zarray": (393, "61cfdce172da056ef17cf80bd9e30d89ddff268f848813b21b488df8e8da270d"),
    "temperature/10957.0.0.0": (7806723, "de314034a300736ff2224f92f8c32f64c66791adfed6cfa4388d6ea99adb744b"),
    "temperature/11048.0.0.0": (7833418, "5f1c3f9545df154d8b55aea5412aa5f4c813e85eb97f9ecdba4ecc7cd30ce38b"),
    "temperature/11185.0.0.0": (7860352, "3c94aa8bd87565bcc925c9bd334db7bfcaac16c1362a857bafe8ac9cdb2c5aad"),
    "temperature/11459.0.0.0": (7808917, "3541ff21a48367c0f4234f5636d95852be6063ccef60d6899a124ae6d42feb18"),
    "u_component_of_wind/.zarray": (393, "61cfdce172da056ef17cf80bd9e30d89ddff268f848813b21b488df8e8da270d"),
    "u_component_of_wind/10957.0.0.0": (10334472, "dba4740ec5d28cb60d2e062480897987851387cd5e0464c85dc06b2a074e9f5d"),
    "u_component_of_wind/11048.0.0.0": (10374965, "7c31a0e5421dcca225d6731eb20e4b62e07cc3d0e4bcc75353d4618e1bc849ee"),
    "u_component_of_wind/11185.0.0.0": (10308600, "b1396833d45d942c4f963305622647cf51b555abfc475f11b3c1516c473dddb8"),
    "u_component_of_wind/11459.0.0.0": (10376135, "c365e4e8fdcb00626693219ed03e32310b3a2bfcb83a0fadf7bfff28414042d2"),
    "v_component_of_wind/.zarray": (393, "61cfdce172da056ef17cf80bd9e30d89ddff268f848813b21b488df8e8da270d"),
    "v_component_of_wind/10957.0.0.0": (10483018, "aa5a30b21d57b56d17bab0f173f4c085cf87f26220def8d099868681a9c30efd"),
    "v_component_of_wind/11048.0.0.0": (10541142, "8fc0d4390f04c7a56875c395ebcd123c32d500682cb527bfe887359cf0cd6219"),
    "v_component_of_wind/11185.0.0.0": (10516124, "5184bd1c8e64b582d74e6efd645b8cd5747431f8caa50e1fab86e5beb14955cc"),
    "v_component_of_wind/11459.0.0.0": (10519849, "1a43b22672a16c5d08ef5a93a13abfdaf659edd22fa365c3a12fb1fdec90b007"),
}


def _sha256(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()


def fetch_object(key: str, *, cache_dir: str | Path | None = None, fetcher: Any = None) -> bytes:
    """Return the bytes of one pinned WeatherBench 2 object, from the cache or the bucket, digest-verified."""
    if key not in WB2_OBJECTS:
        raise ValueError(f"{key} is not a pinned WeatherBench 2 object")
    size, digest = WB2_OBJECTS[key]
    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR
    cache.mkdir(parents=True, exist_ok=True)
    local = cache / key.replace("/", "__")
    if local.is_file():
        data = local.read_bytes()
        if len(data) == size and _sha256(data) == digest:
            return data
    if fetcher is not None:
        data = fetcher(key)
    else:
        with urllib.request.urlopen(f"{WB2_BASE_URL}/{key}", timeout=180) as response:  # noqa: S310 (pinned https URL)
            data = response.read()
    if len(data) != size or _sha256(data) != digest:
        raise ValueError(f"{key}: fetched {len(data)} bytes with sha256 {_sha256(data)[:16]}…, pinned {size} / {digest[:16]}…")
    local.write_bytes(data)
    return data


def _decode(key: str, array_key: str, **kwargs: Any) -> Any:
    """Decode one Zarr v2 chunk (Blosc) into a numpy array shaped like its `.zarray` chunks."""
    import numpy as np
    from numcodecs import Blosc

    meta = json.loads(fetch_object(array_key, **kwargs))
    if meta.get("compressor", {}).get("id") != "blosc" or meta.get("filters") or meta.get("order") != "C":
        raise ValueError(f"{array_key}: unexpected Zarr encoding {meta.get('compressor')}")
    raw = fetch_object(key, **kwargs)
    return np.frombuffer(Blosc().decode(raw), dtype=meta["dtype"]).reshape(meta["chunks"])


def fetch_sample_window(name: str, *, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, Any]:
    """Assemble one named tutorial window (lat 90→-90, lon 0→360, 8 analyses) from pinned objects."""
    if name not in SAMPLE_WINDOWS:
        raise ValueError(f"window must be one of {sorted(SAMPLE_WINDOWS)}")
    import numpy as np

    kw = {"cache_dir": cache_dir, "fetcher": fetcher}
    chunk = SAMPLE_WINDOWS[name]
    lat = _decode("latitude/0", "latitude/.zarray", **kw)
    lon = _decode("longitude/0", "longitude/.zarray", **kw)
    levels = _decode("level/0", "level/.zarray", **kw)
    first = chunk * WB2_CHUNK_STEPS  # the store's time axis is uniform: hours since 1959-01-01 in steps of 6 (verified at build)
    times = [WB2_EPOCH + timedelta(hours=TIMESTEP_HOURS * (first + j)) for j in range(WB2_CHUNK_STEPS)]
    # WB2 stores (time, [level,] longitude, latitude) with latitude ascending; Aurora wants (…, lat, lon), lat descending.
    surf = {
        k: np.ascontiguousarray(_decode(f"{v}/{chunk}.0.0", f"{v}/.zarray", **kw).transpose(0, 2, 1)[:, ::-1, :])
        for k, v in WB2_SURF.items()
    }
    atmos = {
        k: np.ascontiguousarray(_decode(f"{v}/{chunk}.0.0.0", f"{v}/.zarray", **kw).transpose(0, 1, 3, 2)[:, :, ::-1, :])
        for k, v in WB2_ATMOS.items()
    }
    static = {k: np.ascontiguousarray(_decode(f"{v}/0.0", f"{v}/.zarray", **kw).T[::-1, :]) for k, v in WB2_STATIC.items()}
    window = {
        "name": name,
        "lat": lat[::-1].tolist(),
        "lon": lon.tolist(),
        "levels": [int(x) for x in levels],
        "times": [t.strftime("%Y-%m-%dT%H:%M:%S") for t in times],
        "surf": surf,
        "atmos": atmos,
        "static": static,
        "source": SAMPLE_LABEL_SOURCE,
    }
    return _check_window(window) | {"source": SAMPLE_LABEL_SOURCE}


def fetch_sample_dataset(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, Any]]:
    """All four tutorial windows, keyed by role name."""
    return {name: fetch_sample_window(name, cache_dir=cache_dir, fetcher=fetcher) for name in SAMPLE_WINDOWS}


def validate_dataset(
    windows: Sequence[Mapping[str, Any]], *, min_windows: int = MIN_WINDOWS, min_steps: int = HISTORY_STEPS + 1
) -> dict[str, Any]:
    """Structural validation of a list of windows; raises ValueError before any model import."""
    if isinstance(windows, Mapping) or not isinstance(windows, Sequence) or isinstance(windows, (str, bytes)):
        raise ValueError("windows must be a list of window mappings")
    if len(windows) < min_windows:
        raise ValueError(f"{len(windows)} windows; at least {min_windows} are required")
    checked = []
    names: set[str] = set()
    shape = None
    for window in windows:
        c = _check_window(window)
        if c["n_steps"] < min_steps:
            raise ValueError(f"{c['name']}: {c['n_steps']} steps; at least {min_steps} are required")
        if c["name"] in names:
            raise ValueError(f"duplicate window name {c['name']!r}")
        names.add(c["name"])
        if shape is None:
            shape = c["shape"]
        elif c["shape"] != shape:
            raise ValueError(f"{c['name']}: grid {c['shape']} differs from {shape}; all windows must share one grid")
        checked.append(c)
    return {
        "windows": checked,
        "n_windows": len(checked),
        "shape": shape,
        "n_steps": [c["n_steps"] for c in checked],
        "forecast_origins": sum(c["n_steps"] - HISTORY_STEPS for c in checked),
        "time_span": [checked[0]["times"][0], checked[-1]["times"][-1]],
        "digest": dataset_digest(checked),
    }


def dataset_digest(windows: Sequence[Mapping[str, Any]]) -> str:
    return hashlib.sha256("\n".join(window_digest(w) for w in windows).encode("utf-8")).hexdigest()


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read a NetCDF file into one window: coordinates `latitude`/`longitude`/`level`/`time`, surface
    variables as (time, lat, lon), atmospheric variables as (time, level, lat, lon), static fields as
    (lat, lon), all under their Aurora short names (`2t`, `10u`, `10v`, `msl`, `t`, `u`, `v`, `q`, `z`,
    `lsm`, `z_static` or `z` in `static_*`, `slt`) — the shape `write_window_netcdf` produces."""
    import numpy as np
    import xarray as xr

    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    if file_path.suffix.lower() not in (".nc", ".nc4", ".netcdf"):
        raise ValueError("BYOD datasets must be NetCDF (.nc)")
    with xr.open_dataset(file_path) as ds:
        for coord in ("latitude", "longitude", "level", "time"):
            if coord not in ds.coords and coord not in ds.variables:
                raise ValueError(f"NetCDF is missing coordinate {coord!r}")
        times = [datetime.strptime(str(np.datetime_as_string(t, unit="s")), "%Y-%m-%dT%H:%M:%S") for t in ds["time"].values]
        window = {
            "name": file_path.stem,
            "lat": ds["latitude"].values.tolist(),
            "lon": ds["longitude"].values.tolist(),
            "levels": [int(x) for x in ds["level"].values],
            "times": [t.strftime("%Y-%m-%dT%H:%M:%S") for t in times],
            "surf": {k: np.asarray(ds[k].values) for k in SURF_VARS if k in ds},
            "atmos": {k: np.asarray(ds[k].values) for k in ATMOS_VARS if k in ds},
            "static": {k: np.asarray(ds[f"static_{k}"].values) for k in STATIC_VARS if f"static_{k}" in ds},
        }
    return [window]


def write_window_netcdf(window: Mapping[str, Any], path: str | Path) -> Path:
    """Write a validated window as NetCDF in the shape `load_byod_dataset` reads."""
    import numpy as np
    import xarray as xr

    c = _check_window(window)
    coords = {
        "time": np.array([np.datetime64(t) for t in c["times"]]),
        "level": np.array(c["levels"], dtype=np.int64),
        "latitude": np.array(c["lat"], dtype=np.float64),
        "longitude": np.array(c["lon"], dtype=np.float64),
    }
    data = {}
    for k, v in c["surf"].items():
        data[k] = (("time", "latitude", "longitude"), v)
    for k, v in c["atmos"].items():
        data[k] = (("time", "level", "latitude", "longitude"), v)
    for k, v in c["static"].items():
        data[f"static_{k}"] = (("latitude", "longitude"), v)
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    xr.Dataset(data, coords=coords, attrs={"source": str(window.get("source", "")), "timestep_hours": TIMESTEP_HOURS}).to_netcdf(
        out
    )
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `3`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `a96afd7ee6d6…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `AuroraPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), use_lora=True, report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "aurora-0.25-small",
  "modelId": "microsoft/aurora",
  "revision": "a96afd7ee6d65e3bd2d476f3be798a25a56f2296",
  "files": [
    {
      "path": "README.md",
      "bytes": 1152,
      "sha256": "d969382582f6e1eebd21aefe44d776a3ec1b53c351fced08411c3be23def8385"
    },
    {
      "path": "aurora-0.25-small-pretrained.ckpt",
      "bytes": 451339106,
      "sha256": "f80f78de1524a9faba8c9053e4a8ce6a2114ec01cff7f7b4efe9377200d50621",
      "note": "torch archive whose pickle holds a plain state dict (globals: collections.OrderedDict, torch._utils._rebuild_tensor_v2, torch.FloatStorage). Never served: loaded once through the weights-only unpickler, adapted with the upstream compatibility shim and re-saved as aurora-0.25-small-pretrained.safetensors (digest pinned in pipeline.py)."
    },
    {
      "path": "aurora-0.25-static.pickle",
      "bytes": 12459115,
      "sha256": "e382103f6b24bcf1f996cc0af217c71ff2fc66507a5221e1300b5017581bd318",
      "note": "plain pickle of three float32 (721, 1440) numpy arrays lsm/z/slt (globals: numpy.core.numeric._frombuffer, numpy.dtype). Never served: loaded once through a find_class allow-list of those two names and re-saved as aurora-0.25-static.safetensors (digest pinned in pipeline.py)."
    }
  ],
  "totalBytes": 463799373
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = AuroraPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), use_lora=True, report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample windows, validation and roles

The default dataset is real ERA5 reanalysis regridded to 1.5° by WeatherBench 2: four windows of eight 6-hourly analyses — two for training (late December 2018 / early January 2019, and July 2019), one for validation (April 2020) and one for testing (October 2021) — distinct seasons and years, so nothing in the test window is a near-duplicate of anything trained on. `fetch_sample_dataset` retrieves each Zarr chunk, `.zarray` descriptor and coordinate from the pinned table `WB2_OBJECTS` (byte size and SHA-256 per object), refuses a mismatch before decoding, decodes Blosc/LZ4 with `numcodecs`, and reorders the arrays into the Aurora layout (latitude 90 → −90, longitude 0 → 360). `validate_dataset` checks every window and the shared grid before any model runs.

Look for: four windows of 8 steps on a 121 × 240 grid, 24 forecast origins in total, a dataset digest, and a written `outputs/aurora_earth_system_sample_window.nc` (the first three steps of the test window, the NetCDF shape BYOD expects). Four refusal probes follow — wrong pressure levels, an odd longitude count, an implausible temperature field, irregular time spacing — each rejected before `torch` does anything.

In [ ]:
import json
import os
from pathlib import Path

import numpy as np

USE_BYOD = False  # @param {type:"boolean"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    byod_window = load_byod_dataset(byod_path)[0]
    train_windows, val_window, test_window = [byod_window], byod_window, byod_window
    data_source = 'BYOD (' + file_name + ') -- one window plays every role, so the held-out numbers are NOT independent'
else:
    windows = fetch_sample_dataset(cache_dir='weights/wb2-era5-1p5deg')
    train_windows = [windows['train-2019-01'], windows['train-2019-07']]
    val_window, test_window = windows['val-2020-04'], windows['test-2021-10']
    data_source = SAMPLE_LABEL_SOURCE

dataset_manifest = validate_dataset([*train_windows, val_window, test_window] if not USE_BYOD else [test_window])
print({'data_source': data_source, 'n_windows': dataset_manifest['n_windows'], 'grid': dataset_manifest['shape'], 'steps_per_window': dataset_manifest['n_steps'], 'forecast_origins': dataset_manifest['forecast_origins']})
print({'time_span': dataset_manifest['time_span'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'test_window': validate_inputs(test_window)})
print({'2t_mean_K_by_window': {w['name']: round(float(w['surf']['2t'].mean()), 2) for w in [*train_windows, val_window, test_window]}})

def trim_window(window, steps):
    return {**window, 'times': window['times'][:steps], 'surf': {k: v[:steps] for k, v in window['surf'].items()}, 'atmos': {k: v[:steps] for k, v in window['atmos'].items()}}

sample_path = write_window_netcdf(trim_window(test_window, 3), 'outputs/aurora_earth_system_sample_window.nc')
print({'sample_netcdf': str(sample_path), 'megabytes': round(sample_path.stat().st_size / 1e6, 1)})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'wrong pressure levels': {**test_window, 'levels': list(range(13))},
    'odd longitude count': {**test_window, 'lon': test_window['lon'][:-1], 'surf': {k: v[..., :-1] for k, v in test_window['surf'].items()}, 'atmos': {k: v[..., :-1] for k, v in test_window['atmos'].items()}, 'static': {k: v[..., :-1] for k, v in test_window['static'].items()}},
    'implausible temperature': {**test_window, 'surf': {**test_window['surf'], '2t': test_window['surf']['2t'] + 500.0}},
    'irregular time spacing': {**test_window, 'times': test_window['times'][:-1] + ['2030-01-01T00:00:00']},
}
for name, window in probes.items():
    try:
        validate_inputs(window)
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Zero-shot roll-out from the frozen model

`pipe.predict` takes a window, an origin (the index of the latest analysis to use; the step before it is the second history step) and a number of 6-hour steps, and rolls the model forward autoregressively: each forecast becomes the next input. Returned fields are (120, 240) — the model predicts the rows the patch size covers and drops the last latitude row, exactly as it does at 0.25° (721 → 720).

The pipeline was built with `use_lora=True`: the 80 LoRA tensors are zero-initialised, so this is the pretrained model's behaviour. Look for: four valid times 6 h apart, a second call returning bit-identical fields (deterministic on CPU), and global means that stay physical. The mean surface pressure is printed in Pa; 2 m temperature in K.

In [ ]:
import time

ORIGIN = 1  # @param {type:"integer"}
ROLLOUT_STEPS = 4  # @param {type:"integer"}

t0 = time.perf_counter()
rollout_result = pipe.predict(test_window, origin=ORIGIN, steps=ROLLOUT_STEPS)
print({'origin_time': rollout_result['origin_time'], 'shape': rollout_result['shape'], 'seconds': round(time.perf_counter() - t0, 2), 'adapted': rollout_result['model']['adapted']})
for forecast in rollout_result['forecasts']:
    print({'lead_hours': forecast['lead_hours'], 'valid_time': forecast['valid_time'], '2t_mean_K': round(float(forecast['surf']['2t'].mean()), 3), 'msl_mean_Pa': round(float(forecast['surf']['msl'].mean()), 1), 'z500_mean': round(float(forecast['atmos']['z'][7].mean()), 1)})
again = pipe.predict(test_window, origin=ORIGIN, steps=1)
repeat_identical = bool(np.array_equal(again['forecasts'][0]['surf']['2t'], rollout_result['forecasts'][0]['surf']['2t']))
print({'repeat_identical': repeat_identical, 'units': rollout_result['units']})
assert repeat_identical
assert rollout_result['shape'] == (dataset_manifest['shape'][0] - dataset_manifest['shape'][0] % 4, dataset_manifest['shape'][1])

## 6. Persistence baseline and the frozen model's error by lead time

Every number in this notebook is a latitude-weighted RMSE (cos-latitude weights with unit mean, the WeatherBench convention) against the ERA5 analysis at the valid time, averaged over every forecast origin the window allows. The **persistence** forecast carries the latest analysis forward unchanged; at 6 h it is a strong baseline, and for 2 m temperature it gets *better* again at 24 h because the diurnal cycle comes back into phase — a reminder that a baseline is not a straw man. The **skill** column is RMSE(model) / RMSE(persistence): below 1 beats persistence.

Expect the frozen model, run six times coarser than its training grid, to lose to persistence on most variables at 6 h (mean skill above 1) and to close the gap with lead time as persistence decays. This is the out-of-distribution number the adaptation is read against; it says nothing about Aurora at its native resolution, where the published model is far ahead of persistence.

In [ ]:
MAX_LEAD_STEPS = 4  # @param {type:"integer"}

baseline_persistence = persistence_only(test_window, max_lead_steps=MAX_LEAD_STEPS)
print({'persistence_only': {name: {lead: round(row['persistence'], 3) for lead, row in table.items()} for name, table in baseline_persistence['variables'].items() if name in ('2t', 'msl', 'z500')}})
t0 = time.perf_counter()
frozen_test = pipe.evaluate(test_window, max_lead_steps=MAX_LEAD_STEPS)
print({'frozen_model_seconds': round(time.perf_counter() - t0, 1), 'origins': frozen_test['n_origins'], 'metric': frozen_test['metric']})
for name in ('2t', '10u', 'msl', 't', 'z500'):
    print({name: {lead: {'model': round(row['model'], 3), 'persistence': round(row['persistence'], 3), 'skill': round(row['skill'], 3)} for lead, row in frozen_test['variables'][name].items()}})
print({'frozen_summary': {lead: {'mean_skill': round(s['mean_skill'], 3), 'variables_beating_persistence': f"{s['variables_beating_persistence']}/9"} for lead, s in frozen_test['summary'].items()}})

## 7. Bounded LoRA fine-tuning

`pipe.adapt` trains the 80 LoRA tensors (rank 8, 540,672 parameters — 0.5 % of the model) that the upstream architecture places on the query/key/value and output projections of every backbone attention block, and nothing else. Each training sample is one 6-hour forecast from one origin of one training window (12 samples here); the loss is the mean over the nine variables of the MSE divided by a fixed per-variable scale, so a kelvin of temperature and a pascal of pressure count alike; AdamW at a fixed learning rate, seeded shuffling, no scheduler. Epoch 0 records the frozen model (LoRA at zero), and the epoch with the lowest validation loss is kept.

Watch the validation loss fall by two thirds and the 6-hour skill of 2 m temperature and mean sea-level pressure drop below 1 within a few epochs; six epochs take about four minutes on CPU. `TRAINABLE = 'lora+heads'` also unfreezes the encoder token embeddings and decoder heads (713 k parameters) — in the build record it helped the upper air (z500 below persistence at 6 h) and hurt the surface, so LoRA alone is the default.

In [ ]:
EPOCHS = 6  # @param {type:"integer"}
LEARNING_RATE = 1e-3  # @param {type:"number"}
TRAINABLE = 'lora'  # @param ["lora", "lora+heads"]

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_skill_6h'] = {name: round(entry['val'][name]['6h']['skill'], 3) for name in ('2t', 'msl', 'z500')}
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_windows, val_window, epochs=EPOCHS, lr=LEARNING_RATE, trainable=TRAINABLE, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'training_samples': adapt_result['n_train_samples'], 'best_epoch': adapt_result['best_epoch'], 'seconds': adapt_seconds})
print({'loss_scales': adapt_result['loss_scales']})

## 8. Held-out evaluation by lead time

The test window (October 2021) was never used for training or epoch selection. The adapted model is rolled out from every origin to 24 h and scored exactly as the frozen model was in Section 6; the table puts the three numbers side by side per variable and lead. Look for the mean skill dropping below 1 at every lead and most variables beating persistence — the cell asserts that the adapted 6-hour mean skill is below the frozen model's and that more variables beat persistence than before. One two-day window from one seeded run gives no dispersion estimate; these are sample-sanity numbers that show the adaptation contract works, not a benchmark.

In [ ]:
adapted_test = pipe.evaluate(test_window, max_lead_steps=MAX_LEAD_STEPS)
adapted_val = pipe.evaluate(val_window, max_lead_steps=1)
comparison = {}
for name in ('2t', '10u', '10v', 'msl', 't', 'u', 'v', 'q', 'z', 'z500'):
    comparison[name] = {lead: {'persistence': round(row['persistence'], 4), 'frozen': round(frozen_test['variables'][name][lead]['model'], 4), 'adapted': round(row['model'], 4), 'skill_frozen': round(frozen_test['variables'][name][lead]['skill'], 3), 'skill_adapted': round(row['skill'], 3)} for lead, row in adapted_test['variables'][name].items()}
for name in ('2t', 'msl', 'z500', 'u'):
    print({name: comparison[name]})
summary = {lead: {'frozen_mean_skill': round(frozen_test['summary'][lead]['mean_skill'], 3), 'adapted_mean_skill': round(adapted_test['summary'][lead]['mean_skill'], 3), 'beating_persistence_frozen': frozen_test['summary'][lead]['variables_beating_persistence'], 'beating_persistence_adapted': adapted_test['summary'][lead]['variables_beating_persistence']} for lead in adapted_test['leads']}
for lead, row in summary.items():
    print({lead: row})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'roles': {'train': [w['name'] for w in train_windows], 'validation': val_window['name'], 'test': test_window['name']},
    'persistence_baseline': baseline_persistence,
    'frozen_test': frozen_test,
    'validation_metrics': adapted_val,
    'test_metrics': adapted_test,
    'comparison': comparison,
    'summary': summary,
    'adaptation': {k: v for k, v in adapt_result.items() if k not in ('history', 'trainable_names')},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/aurora_earth_system_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert adapted_test['summary']['6h']['mean_skill'] < frozen_test['summary']['6h']['mean_skill']
assert adapted_test['summary']['6h']['variables_beating_persistence'] > frozen_test['summary']['6h']['variables_beating_persistence']
print({'report': 'outputs/aurora_earth_system_evaluation_report.json'})

## 9. Forecast from a new origin, artifact export and fresh reload

The adapted model forecasts 24 h from a later origin of the test window; the forecast valid times, global means and the per-lead 2 m temperature RMSE against the analysis are printed as a sanity check, not an evaluation.

`pipe.save_artifact` writes the trained tensors — the 80 LoRA tensors, about 2 MB — as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digests of the converted base files, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `AuroraPipeline.from_artifact` re-verifies the base files, checks the artifact manifest and digest **before** deserialising, rebuilds the model with LoRA and overlays the tensors — a fresh object from files, not the in-memory model (VER2). The cell asserts identical forecast fields (VER4).

In [ ]:
import platform
import shutil

import safetensors

NEW_ORIGIN = min(test_window['n_steps'] - 1 - ROLLOUT_STEPS, 3)
new_forecast = pipe.predict(test_window, origin=NEW_ORIGIN, steps=ROLLOUT_STEPS)
print({'origin_time': new_forecast['origin_time'], 'adapted': new_forecast['model']['adapted']})
height = new_forecast['shape'][0]
for k, forecast in enumerate(new_forecast['forecasts'], start=1):
    truth = test_window['surf']['2t'][NEW_ORIGIN + k, :height]
    print({'lead_hours': forecast['lead_hours'], 'valid_time': forecast['valid_time'], '2t_mean_K': round(float(forecast['surf']['2t'].mean()), 3), '2t_rmse_vs_analysis': round(lat_weighted_rmse(forecast['surf']['2t'], truth, new_forecast['lat']), 3), 'note': 'sanity check, not an evaluation'})
with open('outputs/aurora_earth_system_forecast.json', 'w', encoding='utf-8') as f:
    json.dump({'origin_time': new_forecast['origin_time'], 'shape': new_forecast['shape'], 'units': new_forecast['units'], 'forecasts': [{'lead_hours': fc['lead_hours'], 'valid_time': fc['valid_time'], 'surf_means': {k: float(v.mean()) for k, v in fc['surf'].items()}} for fc in new_forecast['forecasts']]}, f, indent=2)

artifact_dir = Path('outputs/aurora_earth_system_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'aurora_earth_system', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = AuroraPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = pipe.predict(test_window, origin=ORIGIN, steps=2)['forecasts']
after = reloaded.predict(test_window, origin=ORIGIN, steps=2)['forecasts']
parity = {'max_abs_surf_diff': max(float(np.abs(a['surf'][k] - b['surf'][k]).max()) for a, b in zip(before, after) for k in a['surf']), 'max_abs_atmos_diff': max(float(np.abs(a['atmos'][k] - b['atmos'][k]).max()) for a, b in zip(before, after) for k in a['atmos'])}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['max_abs_surf_diff'] < 1e-6 and parity['max_abs_atmos_diff'] < 1e-6

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_assets': [e for e in MANIFEST['files'] if e['path'] in (SOURCE_CKPT_NAME, SOURCE_STATIC_NAME)],
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickles_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
        'data_objects': len(WB2_OBJECTS),
        'data_base_url': WB2_BASE_URL,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'timm': timm.__version__, 'xarray': xarray.__version__, 'numcodecs': numcodecs.__version__, 'safetensors': safetensors.__version__},
    'data_source': data_source,
    'summary': summary,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/aurora_earth_system_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

Run six times coarser than its training grid, the frozen small checkpoint loses to persistence on most variables at 6 h — an honest number for an out-of-distribution use, and the one this tutorial exists to move. A LoRA fine-tuning of half a million parameters on twelve one-step forecasts brings the mean skill below 1 at every lead to 24 h, with most variables beating persistence, on a window from a different year and season than anything trained on. That is the claim: the adaptation contract moves a foundation weather model onto a grid it was not trained for from two days of data, and the artifact that carries the change is 2 MB.

The test window is two days of one month, the scores come from a single seeded run with no dispersion estimate, and the small checkpoint is the one upstream publishes for testing. So a skill below 1 here says the contract works, not that this model forecasts at any published accuracy, that it is stable beyond 24 h, or that the adaptation transfers to other seasons — none of which this repository exercises. Fine-tuning on a narrow window can also erode the model elsewhere; upstream fine-tunes on years of data with roll-out training, which is out of scope here.

Three things to carry to real data. **Native resolution:** at 0.25° the published model is far ahead of persistence without any adaptation; the frozen numbers here are a resolution story, not an Aurora story. **Independence:** consecutive analyses are near-duplicates — split by period, never by shuffling steps, and read the persistence column before any model number. **Static fields and units:** the model needs the land-sea mask, surface geopotential and soil type on your grid, SI units and the 13 standard levels; a mismatch is silent unless validation catches it.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify two pickled upstream assets, audit and convert them into safetensors without executing anything outside the audited allow-lists, rebuild the model from the installed package, fetch and validate digest-pinned real reanalysis, execute bounded fine-tuning, evaluate against persistence and the frozen model on an independent window at every lead, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or forecast skill beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE = 'lora+heads'` and compare the surface against the upper air; raise `EPOCHS` or `MAX_LEAD_STEPS`; change `ORIGIN`; or bring your own NetCDF window through BYOD and read the persistence column before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/aurora-earth-system-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/aurora-earth-system-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/aurora-earth-system-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/microsoft/aurora (revision `a96afd7ee6d65e3bd2d476f3be798a25a56f2296`)
- Bodnar, C., Bruinsma, W. P., Lucic, A., et al. (2025). A foundation model for the Earth system. Nature 641, 1180–1187: https://doi.org/10.1038/s41586-025-09005-y
- Rasp, S., Hoyer, S., Merose, A., et al. (2024). WeatherBench 2: A benchmark for the next generation of data-driven global weather models. JAMES 16: https://doi.org/10.1029/2023MS004019
- Hersbach, H., et al. (2020). The ERA5 global reanalysis. QJRMS 146, 1999–2049: https://doi.org/10.1002/qj.3803
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)